# 04 Evaluate

This notebook calls the APIM endpoint for each dataset row and then runs built-in Azure AI Evaluation metrics.

## Load Environment and Dataset

In [1]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('../env/.env')
if not env_path.exists():
    raise RuntimeError('env/.env not found. Run 01_deploy_infra.ipynb first.')

load_dotenv(env_path)

EVAL_DATASET_PATH = os.getenv('EVAL_DATASET_PATH', '../data/eval-prompts.jsonl')
EVAL_OUTPUT_PATH = os.getenv('EVAL_OUTPUT_PATH', '../outputs/eval-results.json')
APIM_ENDPOINT = os.getenv('APIM_ENDPOINT', '').rstrip('/')
APIM_PATH = os.getenv('APIM_CHAT_COMPLETIONS_PATH', '/openai/chat/completions')
APIM_API_KEY = os.getenv('APIM_API_KEY', '')

AOAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '').rstrip('/')
AOAI_KEY = os.getenv('AZURE_OPENAI_API_KEY', '')
AOAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'chat4o')

dataset_path = Path(EVAL_DATASET_PATH)
if not dataset_path.exists():
    raise RuntimeError(f'Dataset file not found: {dataset_path}')

dataset = [json.loads(line) for line in dataset_path.read_text(encoding='utf-8').splitlines() if line.strip()]
print(f'Loaded dataset rows: {len(dataset)}')

Loaded dataset rows: 5


## Generate Responses Through APIM

In [2]:
import requests

if not APIM_API_KEY or APIM_API_KEY.startswith('[Retrieve from Azure Portal'):
    raise RuntimeError('APIM_API_KEY is not set with a real key in env/.env')

url = f"{APIM_ENDPOINT}{APIM_PATH}"
headers = {
    'Ocp-Apim-Subscription-Key': APIM_API_KEY,
    'Content-Type': 'application/json'
}

eval_rows = []
for row in dataset:
    prompt = row.get('query', '')
    payload = {
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 140
    }

    resp = requests.post(url, headers=headers, json=payload, timeout=45)
    if resp.status_code == 200:
        data = resp.json()
        model_response = data.get('choices', [{}])[0].get('message', {}).get('content', '')
    else:
        model_response = f'ERROR {resp.status_code}: {resp.text[:300]}'

    eval_rows.append({
        'query': prompt,
        'response': model_response,
        'context': row.get('expected_behavior', '')
    })

print(f'Collected APIM responses: {len(eval_rows)}')

Collected APIM responses: 5


## Run Azure AI Evaluation

In [4]:
import tempfile
import time
from azure.ai.evaluation import CoherenceEvaluator, FluencyEvaluator, RelevanceEvaluator, evaluate

model_config = {
    'azure_endpoint': AOAI_ENDPOINT,
    'api_key': AOAI_KEY,
    'azure_deployment': AOAI_DEPLOYMENT,
    'api_version': '2024-08-01-preview'
}

relevance = RelevanceEvaluator(model_config=model_config)
coherence = CoherenceEvaluator(model_config=model_config)
fluency = FluencyEvaluator(model_config=model_config)

max_retries = 4
base_backoff_seconds = 20

with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False, encoding='utf-8') as tmp:
    for item in eval_rows:
        tmp.write(json.dumps(item) + '\n')
    temp_path = tmp.name

def run_evaluation_with_retries():
    for attempt in range(1, max_retries + 1):
        try:
            return evaluate(
                data=temp_path,
                evaluators={
                    'relevance': relevance,
                    'coherence': coherence,
                    'fluency': fluency
                },
                evaluator_config={
                    'default': {
                        'column_mapping': {
                            'query': '${data.query}',
                            'response': '${data.response}',
                            'context': '${data.context}'
                        }
                    }
                }
            )
        except Exception as ex:
            msg = str(ex).lower()
            is_rate_limited = '429' in msg or 'too_many_requests' in msg
            if is_rate_limited and attempt < max_retries:
                wait_seconds = base_backoff_seconds * attempt
                print(f"Rate limited, retrying full evaluation in {wait_seconds}s (attempt {attempt}/{max_retries})")
                time.sleep(wait_seconds)
                continue
            raise

raw_result = run_evaluation_with_retries()
result = {
    'metrics': raw_result.get('metrics', {}),
    'rows': len(eval_rows),
    'model_config': {
        'azure_endpoint': AOAI_ENDPOINT,
        'azure_deployment': AOAI_DEPLOYMENT,
        'api_version': '2024-08-01-preview'
    }
}

print('Evaluation complete')
print(json.dumps(result.get('metrics', {}), indent=2))

## Save Results

In [ ]:
output_file = Path(EVAL_OUTPUT_PATH)
output_file.parent.mkdir(parents=True, exist_ok=True)
output_file.write_text(json.dumps(result, indent=2), encoding='utf-8')
print(f'Saved evaluation output: {output_file}')